# Task 4: Portfolio Optimization — Modern Portfolio Theory

**GMF Investments — Efficient Frontier for TSLA, BND, SPY**

**Author:** Sosina Ayele

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import os, pickle, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

try:
    from pypfopt import EfficientFrontier, risk_models, expected_returns
    from pypfopt.plotting import plot_efficient_frontier
    HAS_PYPFOPT = True
    print('PyPortfolioOpt loaded!')
except:
    HAS_PYPFOPT = False
    print('PyPortfolioOpt not available — using scipy')

from scipy.optimize import minimize
print('Libraries loaded!')

## 1. Load Data & TSLA Forecast Return

In [ ]:
# Load prices
csv_path = '../data/processed/prices.csv'
if os.path.exists(csv_path):
    prices = pd.read_csv(csv_path, index_col=0, parse_dates=True)
else:
    raw = yf.download(['TSLA','BND','SPY'], start='2015-01-01', end='2026-06-30', auto_adjust=True)
    prices = raw['Close'].ffill().bfill()

returns = prices.pct_change().dropna()
print(f'Returns shape: {returns.shape}')

# Load TSLA forecast
forecast_path = '../data/processed/tsla_forecast.csv'
if os.path.exists(forecast_path):
    tsla_forecast = pd.read_csv(forecast_path, index_col=0, parse_dates=True)['forecast']
    tsla_ann_return = (tsla_forecast.iloc[-1] / prices['TSLA'].iloc[-1] - 1)
    print(f'TSLA forecasted 6-month return: {tsla_ann_return*100:.2f}%')
    tsla_ann_return_annualized = (1 + tsla_ann_return)**2 - 1
    print(f'TSLA annualized expected return: {tsla_ann_return_annualized*100:.2f}%')
else:
    # Fallback: use historical mean
    tsla_ann_return_annualized = returns['TSLA'].mean() * 252
    print(f'Using historical TSLA return: {tsla_ann_return_annualized*100:.2f}%')

# Historical returns for BND and SPY (annualized)
bnd_ann_return = returns['BND'].mean() * 252
spy_ann_return = returns['SPY'].mean() * 252
print(f'BND historical annual return: {bnd_ann_return*100:.2f}%')
print(f'SPY historical annual return: {spy_ann_return*100:.2f}%')

## 2. Expected Returns & Covariance Matrix

In [ ]:
# Expected returns vector
mu = np.array([tsla_ann_return_annualized, bnd_ann_return, spy_ann_return])
assets = ['TSLA', 'BND', 'SPY']
print('Expected Annual Returns:')
for a, r in zip(assets, mu):
    print(f'  {a}: {r*100:.2f}%')

# Covariance matrix from historical daily returns (annualized)
cov_matrix = returns[assets].cov() * 252
print(f'\nCovariance Matrix (annualized):')
print(cov_matrix.round(6))

# Visualize covariance
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cov_matrix, annot=True, fmt='.4f', cmap='RdYlGn_r',
            ax=ax, square=True)
ax.set_title('Annualized Covariance Matrix — TSLA, BND, SPY', fontweight='bold')
plt.tight_layout()
plt.savefig('covariance_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Covariance matrix saved!')

## 3. Generate Efficient Frontier via Monte Carlo Simulation

In [ ]:
N_PORTFOLIOS = 10000
RISK_FREE = 0.05
np.random.seed(42)

port_returns = []
port_vols = []
port_sharpes = []
port_weights = []

for _ in range(N_PORTFOLIOS):
    w = np.random.dirichlet(np.ones(3))
    r = np.dot(w, mu)
    v = np.sqrt(w @ cov_matrix.values @ w)
    s = (r - RISK_FREE) / v
    port_returns.append(r)
    port_vols.append(v)
    port_sharpes.append(s)
    port_weights.append(w)

port_returns = np.array(port_returns)
port_vols = np.array(port_vols)
port_sharpes = np.array(port_sharpes)
port_weights = np.array(port_weights)

print(f'Simulated {N_PORTFOLIOS:,} portfolios')
print(f'Return range: {port_returns.min()*100:.2f}% to {port_returns.max()*100:.2f}%')
print(f'Vol range: {port_vols.min()*100:.2f}% to {port_vols.max()*100:.2f}%')

## 4. Find Key Portfolios & Plot Efficient Frontier

In [ ]:
# Maximum Sharpe Ratio portfolio
max_sharpe_idx = np.argmax(port_sharpes)
max_sharpe_weights = port_weights[max_sharpe_idx]
max_sharpe_return = port_returns[max_sharpe_idx]
max_sharpe_vol = port_vols[max_sharpe_idx]
max_sharpe_ratio = port_sharpes[max_sharpe_idx]

# Minimum Volatility portfolio
min_vol_idx = np.argmin(port_vols)
min_vol_weights = port_weights[min_vol_idx]
min_vol_return = port_returns[min_vol_idx]
min_vol_vol = port_vols[min_vol_idx]
min_vol_sharpe = port_sharpes[min_vol_idx]

print('=== MAX SHARPE RATIO PORTFOLIO ===')
for a, w in zip(assets, max_sharpe_weights):
    print(f'  {a}: {w*100:.1f}%')
print(f'  Expected Return: {max_sharpe_return*100:.2f}%')
print(f'  Volatility: {max_sharpe_vol*100:.2f}%')
print(f'  Sharpe Ratio: {max_sharpe_ratio:.3f}')

print('\n=== MINIMUM VOLATILITY PORTFOLIO ===')
for a, w in zip(assets, min_vol_weights):
    print(f'  {a}: {w*100:.1f}%')
print(f'  Expected Return: {min_vol_return*100:.2f}%')
print(f'  Volatility: {min_vol_vol*100:.2f}%')
print(f'  Sharpe Ratio: {min_vol_sharpe:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

sc = ax.scatter(port_vols*100, port_returns*100, c=port_sharpes,
                cmap='viridis', alpha=0.4, s=8, zorder=1)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

# Max Sharpe
ax.scatter(max_sharpe_vol*100, max_sharpe_return*100,
           marker='*', s=400, c='red', zorder=5,
           label=f'Max Sharpe ({max_sharpe_ratio:.2f})')
ax.annotate(f'Max Sharpe\nTSLA:{max_sharpe_weights[0]*100:.0f}% BND:{max_sharpe_weights[1]*100:.0f}% SPY:{max_sharpe_weights[2]*100:.0f}%',
            (max_sharpe_vol*100, max_sharpe_return*100),
            textcoords='offset points', xytext=(12, -20),
            fontsize=9, color='red', fontweight='bold')

# Min Vol
ax.scatter(min_vol_vol*100, min_vol_return*100,
           marker='D', s=200, c='blue', zorder=5,
           label=f'Min Volatility ({min_vol_vol*100:.1f}%)')
ax.annotate(f'Min Vol\nTSLA:{min_vol_weights[0]*100:.0f}% BND:{min_vol_weights[1]*100:.0f}% SPY:{min_vol_weights[2]*100:.0f}%',
            (min_vol_vol*100, min_vol_return*100),
            textcoords='offset points', xytext=(8, 8),
            fontsize=9, color='blue', fontweight='bold')

ax.set_xlabel('Annual Volatility (%)', fontsize=12)
ax.set_ylabel('Annual Expected Return (%)', fontsize=12)
ax.set_title('Efficient Frontier — TSLA, BND, SPY Portfolio\n(10,000 Monte Carlo Simulations)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Efficient frontier saved!')

## 5. Portfolio Recommendation

In [ ]:
print('=== GMF INVESTMENTS — PORTFOLIO RECOMMENDATION ===')
print()
print('RECOMMENDED: Maximum Sharpe Ratio Portfolio')
print('(Best risk-adjusted return for GMF clients seeking growth)')
print()
print('Portfolio Weights:')
for a, w in zip(assets, max_sharpe_weights):
    print(f'  {a:5s}: {w*100:6.1f}%')
print()
print(f'Expected Annual Return:  {max_sharpe_return*100:.2f}%')
print(f'Expected Annual Volatility: {max_sharpe_vol*100:.2f}%')
print(f'Sharpe Ratio:  {max_sharpe_ratio:.3f}')
print()
print('Justification:')
print('The Maximum Sharpe Ratio portfolio maximizes risk-adjusted return.')
print('For GMF clients with moderate-to-high risk tolerance, this portfolio')
print('provides the optimal balance between TSLA growth potential,')
print('SPY diversification, and BND stability.')

# Save weights
import json
recommended = {
    'TSLA': float(max_sharpe_weights[0]),
    'BND':  float(max_sharpe_weights[1]),
    'SPY':  float(max_sharpe_weights[2]),
    'expected_return': float(max_sharpe_return),
    'volatility': float(max_sharpe_vol),
    'sharpe': float(max_sharpe_ratio)
}
with open('../data/processed/optimal_weights.json', 'w') as f:
    json.dump(recommended, f, indent=2)
print('\nWeights saved to optimal_weights.json')